<a href="https://colab.research.google.com/github/alexandrequirinodemelo/alexandrequirino/blob/main/script_refatorado_final_v13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# BIBLIOTECAS E PACOTES

!pip install -q keras-tuner

import pandas as pd
import numpy as np
import os
import warnings
import re
import shutil
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)

# --- CONFIGURAÇÕES ---
USAR_CHUVA       = True
MAX_LAG          = 7
CICLOS_VALIDACAO = [2023, 2024, 2025]
GRADE_DEPTH      = [2, 3, 4, 5, 6, 8, 10] # Grade recomendada no Item 21

def carregar_dados_v14_completo():
    df_matriz = pd.read_excel("produtividade_novo_encolding.xlsx")
    col_ano = df_matriz.columns[0]
    df_longo = df_matriz.melt(id_vars=[col_ano], var_name='estado', value_name='produtividade')
    df_longo.rename(columns={col_ano: 'ano_int'}, inplace=True)

    if 2025 not in df_longo['ano_int'].unique():
        print("--> [Aviso] Ano 2025 não encontrado na planilha de produtividade. Estruturando linhas para 2025...")
        estados = df_longo['estado'].unique()
        df_2025 = pd.DataFrame({'ano_int': [2025]*len(estados), 'estado': estados, 'produtividade': [np.nan]*len(estados)})
        df_base = pd.concat([df_longo, df_2025])
    else:
        print("--> [Sucesso] Dados reais de 2025 detectados diretamente na planilha original!")
        df_base = df_longo.copy()

    df_base = df_base.sort_values(['estado', 'ano_int']).reset_index(drop=True)

    for i in range(1, MAX_LAG + 1):
        df_base[f'Prod_t_{i}'] = df_base.groupby('estado')['produtividade'].shift(i)

    if USAR_CHUVA:
        df_c = pd.read_excel("chuva_consolidada_2005_2025_corrigida.xlsx")
        df_c['ano_int'] = df_c['ano'].apply(lambda x: int(re.search(r'(\d{4})', str(x)).group(1)))
        df_c = df_c.groupby(['uf', 'ano_int'])[['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']].mean().reset_index()
        df_base = pd.merge(df_base, df_c.rename(columns={'uf':'estado'}), on=['estado', 'ano_int'], how='left').fillna(0)
    return df_base

def build_dynamic_model_v41(hp, max_lags, n_clima, hybrid=False):
    n_lags_selected = hp.Int('selected_lags', 1, max_lags)
    inputs = layers.Input(shape=(max_lags + n_clima,))
    def slice_inputs(x):
        lags_part = x[:, :n_lags_selected]
        if n_clima > 0:
            clima_part = x[:, max_lags:]
            return tf.concat([lags_part, clima_part], axis=1)
        return lags_part
    x = layers.Lambda(slice_inputs)(inputs)
    x = layers.Reshape((-1, 1))(x)
    if hybrid:
        x = layers.LSTM(hp.Int('u', 32, 64), return_sequences=True)(x)
        x = layers.GRU(hp.Int('g', 16, 32))(x)
    else:
        x = layers.LSTM(hp.Int('u', 32, 64))(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

def otimizar_parametros_tabulares_passado(df_historico, ano_alvo, mod, usar_chuva):
    """
    Item 21: Realiza uma busca em grade aninhada no tempo (Lags x Max_Depth)
    testando estritamente no ano anterior (T-1) para controlar o overfitting.
    """
    ano_val = ano_alvo - 1
    df_train_val = df_historico[(df_historico['ano_int'] < ano_val) & (df_historico['ano_int'] >= 2013) & (df_historico['produtividade'] > 0)]
    df_test_val  = df_historico[df_historico['ano_int'] == ano_val]

    if len(df_test_val) == 0:
        return 1, 4 # Fallbacks seguros (Lag=1, Depth=4)

    best_l = 1
    best_d = 4
    best_mae = float('inf')

    # Varre a combinação de janelas temporais e profundidade das árvores
    for l in range(1, MAX_LAG + 1):
        for d in GRADE_DEPTH:
            f_tab = [f'Prod_t_{i}' for i in range(1, l + 1)] + (['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm'] if usar_chuva else [])
            X_tr_v, X_ts_v = df_train_val[f_tab], df_test_val[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_train_val['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=d, random_state=42, n_jobs=-1).fit(X_tr_v, df_train_val['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_train_val['produtividade'])

            p_v = m_t.predict(X_ts_v)
            mae_v = mean_absolute_error(df_test_val['produtividade'], p_v)

            if mae_v < best_mae:
                best_mae = mae_v
                best_l = l
                best_d = d

    return best_l, best_d

# --- EXECUÇÃO ---
df_base = carregar_dados_v14_completo()
n_clima = 3 if USAR_CHUVA else 0
resumo_metricas, resumo_hparams, previsoes_finais = [], [], []
modelos_lista = ['LSTM', 'LSTM-GRU', 'XGBoost', 'Random Forest', 'Decision Tree', 'Prophet', 'SNaive']

for ano_alvo in CICLOS_VALIDACAO:
    print(f"\n>>> PROCESSANDO CICLO: {ano_alvo}")
    df_train = df_base[(df_base['ano_int'] < ano_alvo) & (df_base['ano_int'] >= 2013) & (df_base['produtividade'] > 0)].copy()
    df_test  = df_base[df_base['ano_int'] == ano_alvo].copy()

    if len(df_test) == 0:
        print(f"  [Aviso] Pulando ciclo {ano_alvo} pois não há amostras de teste.")
        continue

    df_res = df_test[['ano_int', 'estado', 'produtividade']].rename(columns={'produtividade': 'Real'})

    feats_full = [f'Prod_t_{i}' for i in range(1, MAX_LAG + 1)]
    if USAR_CHUVA: feats_full += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

    sc_x = StandardScaler().fit(df_train[feats_full])
    sc_y = StandardScaler().fit(df_train[['produtividade']])
    X_tr = sc_x.transform(df_train[feats_full]); X_ts = sc_x.transform(df_test[feats_full])
    y_tr = sc_y.transform(df_train[['produtividade']]).flatten()

    for mod in modelos_lista:
        print(f"  .. {mod}")

        if mod in ['LSTM', 'LSTM-GRU']:
            if os.path.exists('t_dir'): shutil.rmtree('t_dir')
            tuner = kt.RandomSearch(
                lambda hp: build_dynamic_model_v41(hp, MAX_LAG, n_clima, (mod=='LSTM-GRU')),
                objective=kt.Objective('val_loss', direction='min'),
                max_trials=2,
                directory='t_dir'
            )
            callback_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
            tuner.search(X_tr, y_tr, epochs=150, validation_split=0.15, callbacks=[callback_early], verbose=0)

            m = tuner.get_best_models(1)[0]
            hp_b = tuner.get_best_hyperparameters(1)[0]
            p = sc_y.inverse_transform(m.predict(X_ts, verbose=0)).flatten()
            res_lag, config_str = hp_b.get('selected_lags'), str(hp_b.values)

        elif mod in ['XGBoost', 'Random Forest', 'Decision Tree']:
            # Item 21: Busca dinâmica otimizada trazendo o par perfeito de Lag e Profundidade
            best_l, best_d = otimizar_parametros_tabulares_passado(df_base, ano_alvo, mod, USAR_CHUVA)
            f_tab = [f'Prod_t_{i}' for i in range(1, best_l + 1)] + (['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm'] if USAR_CHUVA else [])
            X_tr_t, X_ts_t = df_train[f_tab], df_test[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=best_d, random_state=42).fit(X_tr_t, df_train['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=best_d, random_state=42, n_jobs=-1).fit(X_tr_t, df_train['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=best_d, random_state=42).fit(X_tr_t, df_train['produtividade'])

            p = m_t.predict(X_ts_t)
            res_lag, config_str = best_l, f"Lag_{best_l}_MaxDepth_{best_d}"

        elif mod == 'Prophet':
            dict_p = {e: Prophet().fit(df_train[df_train['estado']==e][['ano_int','produtividade']].rename(columns={'ano_int':'ds','produtividade':'y'}).assign(ds=lambda x: pd.to_datetime(x.ds, format='%Y'))).predict(pd.DataFrame({'ds':[pd.to_datetime(ano_alvo, format='%Y')]}))['yhat'].values[0] for e in df_test['estado'].unique()}
            p = df_test['estado'].map(dict_p).values; res_lag, config_str = 0, "Univariado"

        elif mod == 'SNaive':
            p = df_test['Prod_t_1'].values; res_lag, config_str = 1, "Naive Lag 1"

        df_res[f'Prev_{mod}'] = np.maximum(p, 0)

        rv = df_res['Real'].values
        if not np.isnan(rv).all():
            resumo_metricas.append({
                'Ano': ano_alvo, 'Modelo': mod,
                'RMSE': np.sqrt(mean_squared_error(rv, p)),
                'MAE': mean_absolute_error(rv, p),
                'MAPE': np.mean(np.abs((rv - p) / np.where(rv==0, 1, rv))) * 100,
                'R2': r2_score(rv, p), 'Lag': res_lag
            })

        resumo_hparams.append({'Ano': ano_alvo, 'Modelo': mod, 'Lag_Final': res_lag, 'Config_Detalhada': config_str})

    previsoes_finais.append(df_res)

# SALVAMENTO
with pd.ExcelWriter("RESULTADO_FINAL_3_ABAS.xlsx") as writer:
    if len(previsoes_finais) > 0:
        pd.concat(previsoes_finais).to_excel(writer, sheet_name='Previsoes', index=False)
    if len(resumo_metricas) > 0:
        pd.DataFrame(resumo_metricas).to_excel(writer, sheet_name='Metricas', index=False)
    pd.DataFrame(resumo_hparams).to_excel(writer, sheet_name='Hiperparametros', index=False)

print("\n>>> SUCESSO! Item 21 integrado. Profundidades de árvores tunadas e gravadas na aba de hiperparâmetros.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.5 MB/s eta 0:00:00
--> [Aviso] Ano 2025 não encontrado na planilha de produtividade. Estruturando linhas para 2025...

>>> PROCESSANDO CICLO: 2023
  .. LSTM
  .. LSTM-GRU
  .. XGBoost
  .. Random Forest
  .. Decision Tree
  .. Prophet
  .. SNaive

>>> PROCESSANDO CICLO: 2024
  .. LSTM
  .. LSTM-GRU
  .. XGBoost
  .. Random Forest
  .. Decision Tree
  .. Prophet
  .. SNaive

>>> PROCESSANDO CICLO: 2025
  .. LSTM


  .. LSTM-GRU


  .. XGBoost
  .. Random Forest
  .. Decision Tree
  .. Prophet
  .. SNaive

>>> SUCESSO! Item 21 integrado. Profundidades de árvores tunadas e gravadas na aba de hiperparâmetros.


In [4]:
# ==============================================================================
# BIBLIOTECAS E PACOTES
# ==============================================================================
!pip install -q keras-tuner

import pandas as pd
import numpy as np
import os
import warnings
import re
import shutil
import logging
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# Supressão de logs desnecessários
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

# ==============================================================================
# CONFIGURAÇÕES GLOBAIS
# ==============================================================================
USAR_CHUVA       = False
MAX_LAG          = 7
CICLOS_VALIDACAO = [2023, 2024, 2025]
GRADE_DEPTH      = [2, 3, 4, 5, 6, 8, 10]

# ==============================================================================
# FUNÇÕES DE CARREGAMENTO E PRÉ-PROCESSAMENTO
# ==============================================================================
def carregar_dados_completo():
    # 1. Carregar Matriz de Produtividade
    df_matriz = pd.read_excel("produtividade_novo_encolding.xlsx")
    col_ano = df_matriz.columns[0]

    # Identificar se há uma coluna de encoding do estado salva na estrutura original
    # Caso não exista explicitamente, o script usará a própria string mapeada internamente.
    col_encoding = [c for c in df_matriz.columns if 'encod' in c.lower() or 'id' in c.lower()]
    col_id_estado = col_encoding[0] if col_encoding else None

    df_longo = df_matriz.melt(id_vars=[col_ano], var_name='estado', value_name='produtividade')
    df_longo.rename(columns={col_ano: 'ano_int'}, inplace=True)

    # Tratar a projeção/estrutura para o ano de 2025
    if 2025 not in df_longo['ano_int'].unique():
        print("--> [Aviso] Ano 2025 estruturado artificialmente para previsão de safra futura.")
        estados = df_longo['estado'].unique()
        df_2025 = pd.DataFrame({'ano_int': [2025]*len(estados), 'estado': estados, 'produtividade': [np.nan]*len(estados)})
        df_base = pd.concat([df_longo, df_2025])
    else:
        print("--> [Sucesso] Dados reais de 2025 detectados na planilha original.")
        df_base = df_longo.copy()

    df_base = df_base.sort_values(['estado', 'ano_int']).reset_index(drop=True)

    # 2. Geração das Variáveis de Lag (Defasagem Temporal)
    for i in range(1, MAX_LAG + 1):
        df_base[f'Prod_t_{i}'] = df_base.groupby('estado')['produtividade'].shift(i)

    # 3. Integração e Tratamento Inteligente dos Dados de Chuva
    if USAR_CHUVA:
        df_c = pd.read_excel("chuva_consolidada_2005_2025_corrigida.xlsx")
        df_c['ano_int'] = df_c['ano'].apply(lambda x: int(re.search(r'(\d{4})', str(x)).group(1)))
        df_c = df_c.groupby(['uf', 'ano_int'])[['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']].mean().reset_index()

        df_base = pd.merge(df_base, df_c.rename(columns={'uf':'estado'}), on=['estado', 'ano_int'], how='left')

        # Correção do Gap Oculto: Imputação pela MEDIANA histórica do próprio estado, evitando viés do zero
        colunas_chuva = ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']
        for col in colunas_chuva:
            df_base[col] = df_base.groupby('estado')[col].transform(lambda x: x.fillna(x.median()))
            df_base[col] = df_base[col].fillna(0) # Fallback absoluto caso o estado não tenha histórico algum

    # Mapeamento numérico simples para o Estado (garantindo inteligência geográfica aos modelos globais)
    df_base['estado_encoded'] = df_base['estado'].astype('category').cat.codes

    return df_base

# ==============================================================================
# ARQUITETURA DINÂMICA DE REDES NEURAIS (KERAS)
# ==============================================================================
def build_dynamic_model_v42(hp, max_lags, n_clima, hybrid=False):
    n_lags_selected = hp.Int('selected_lags', 2, max_lags)
    inputs = layers.Input(shape=(max_lags + n_clima,))

    # Fatiamento cirúrgico dos tensores para alinhar Lags e Variáveis Climáticas Corretamente
    def slice_inputs(x):
        lags_part = x[:, :n_lags_selected]
        if n_clima > 0:
            clima_part = x[:, max_lags:]
            return tf.concat([lags_part, clima_part], axis=1)
        return lags_part

    x = layers.Lambda(slice_inputs)(inputs)
    x = layers.Reshape((-1, 1))(x)

    if hybrid:
        x = layers.LSTM(hp.Int('u', 32, 128, step=32), return_sequences=True)(x)
        x = layers.GRU(hp.Int('g', 16, 64, step=16))(x)
    else:
        x = layers.LSTM(hp.Int('u', 32, 128, step=32))(x)

    x = layers.Dropout(hp.Float('drop', 0.0, 0.3, step=0.1))(x)
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('lr', [1e-3, 5e-4])), loss='mse')
    return model

# ==============================================================================
# OTIMIZAÇÃO EM GRADE ANINHADA NO TEMPO (MODELOS TABULARES)
# ==============================================================================
def otimizar_parametros_tabulares_passado(df_historico, ano_alvo, mod, usar_chuva):
    ano_val = ano_alvo - 1
    df_train_val = df_historico[(df_historico['ano_int'] < ano_val) & (df_historico['ano_int'] >= 2013) & (df_historico['produtividade'] > 0)]
    df_test_val  = df_historico[df_historico['ano_int'] == ano_val]

    if len(df_test_val) == 0:
        return 1, 4 # Fallback estável de segurança

    best_l, best_d, best_mae = 1, 4, float('inf')

    for l in range(1, MAX_LAG + 1):
        for d in GRADE_DEPTH:
            f_tab = ['estado_encoded'] + [f'Prod_t_{i}' for i in range(1, l + 1)]
            if usar_chuva:
                f_tab += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

            X_tr_v = df_train_val[f_tab]
            X_ts_v = df_test_val[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=d, random_state=42, n_estimators=100).fit(X_tr_v, df_train_val['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=d, random_state=42, n_jobs=-1, n_estimators=100).fit(X_tr_v, df_train_val['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_train_val['produtividade'])

            p_v = m_t.predict(X_ts_v)
            mae_v = mean_absolute_error(df_test_val['produtividade'], p_v)

            if mae_v < best_mae:
                best_mae, best_l, best_d = mae_v, l, d

    return best_l, best_d

# ==============================================================================
# PIPELINE PRINCIPAL DE EXECUÇÃO
# ==============================================================================
df_base = carregar_dados_completo()
n_clima = 3 if USAR_CHUVA else 0
resumo_metricas, resumo_hparams, previsoes_finais = [], [], []
modelos_lista = ['LSTM', 'LSTM-GRU', 'XGBoost', 'Random Forest', 'Decision Tree', 'Prophet', 'SNaive']

for ano_alvo in CICLOS_VALIDACAO:
    print(f"\n>>> PROCESSANDO CICLO (JANELA EXPANSIVA): {ano_alvo}")

    df_train = df_base[(df_base['ano_int'] < ano_alvo) & (df_base['ano_int'] >= 2013) & (df_base['produtividade'] > 0)].copy()
    df_test  = df_base[df_base['ano_int'] == ano_alvo].copy()

    if len(df_test) == 0:
        print(f"  [Aviso] Pulando ciclo {ano_alvo}. Sem dados de validação.")
        continue

    df_res = df_test[['ano_int', 'estado', 'produtividade']].rename(columns={'produtividade': 'Real'})

    # Definição e Ajuste Escalar Robusto das Redes Neurais
    feats_full = [f'Prod_t_{i}' for i in range(1, MAX_LAG + 1)]
    if USAR_CHUVA:
        feats_full += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

    sc_x = StandardScaler().fit(df_train[feats_full])
    sc_y = StandardScaler().fit(df_train[['produtividade']])

    X_tr = sc_x.transform(df_train[feats_full])
    X_ts = sc_x.transform(df_test[feats_full])
    y_tr = sc_y.transform(df_train[['produtividade']]).flatten()

    for mod in modelos_lista:
        print(f"  .. Executando: {mod}")

        if mod in ['LSTM', 'LSTM-GRU']:
            dir_tuner = f't_dir_{mod}_{ano_alvo}'
            if os.path.exists(dir_tuner): shutil.rmtree(dir_tuner)

            tuner = kt.RandomSearch(
                lambda hp: build_dynamic_model_v42(hp, MAX_LAG, n_clima, (mod=='LSTM-GRU')),
                objective=kt.Objective('val_loss', direction='min'),
                max_trials=7, # Aumentado para garantir exploração mínima do espaço de hiperparâmetros
                directory=dir_tuner,
                project_name='tuning'
            )

            callback_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            tuner.search(X_tr, y_tr, epochs=120, validation_split=0.15, callbacks=[callback_early], verbose=0)

            m = tuner.get_best_models(1)[0]
            hp_b = tuner.get_best_hyperparameters(1)[0]
            p = sc_y.inverse_transform(m.predict(X_ts, verbose=0)).flatten()
            res_lag, config_str = hp_b.get('selected_lags'), str(hp_b.values)

            shutil.rmtree(dir_tuner) # Limpeza de arquivos temporários após o ciclo

        elif mod in ['XGBoost', 'Random Forest', 'Decision Tree']:
            best_l, best_d = otimizar_parametros_tabulares_passado(df_base, ano_alvo, mod, USAR_CHUVA)

            # Reconstruindo as colunas incluindo explicitamente a identidade do estado codificado
            f_tab = ['estado_encoded'] + [f'Prod_t_{i}' for i in range(1, best_l + 1)]
            if USAR_CHUVA:
                f_tab += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

            X_tr_t = df_train[f_tab]
            X_ts_t = df_test[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=best_d, random_state=42, n_estimators=150).fit(X_tr_t, df_train['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=best_d, random_state=42, n_jobs=-1, n_estimators=150).fit(X_tr_t, df_train['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=best_d, random_state=42).fit(X_tr_t, df_train['produtividade'])

            p = m_t.predict(X_ts_t)
            res_lag, config_str = best_l, f"Lag_{best_l}_MaxDepth_{best_d}"

        elif mod == 'Prophet':
            dict_p = {}
            for e in df_test['estado'].unique():
                df_e = df_train[df_train['estado'] == e][['ano_int', 'produtividade', 'Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']].copy()
                df_e = df_e.rename(columns={'ano_int': 'ds', 'produtividade': 'y'})
                df_e['ds'] = pd.to_datetime(df_e['ds'], format='%Y')

                # Instanciando o Prophet com variáveis exógenas climáticas (Aprimoramento Multivariado)
                m_prophet = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
                if USAR_CHUVA:
                    m_prophet.add_regressor('Chuva_Q1_mm')
                    m_prophet.add_regressor('Chuva_Q2_mm')
                    m_prophet.add_regressor('Chuva_Q3_mm')

                m_prophet.fit(df_e)

                df_fut = pd.DataFrame({'ds': [pd.to_datetime(ano_alvo, format='%Y')]})
                if USAR_CHUVA:
                    df_fut_chuva = df_test[df_test['estado'] == e][['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']]
                    df_fut = pd.concat([df_fut, df_fut_chuva.reset_index(drop=True)], axis=1)

                pred_p = m_prophet.predict(df_fut)
                dict_p[e] = pred_p['yhat'].values[0]

            p = df_test['estado'].map(dict_p).values
            res_lag, config_str = 0, "Multivariado_Exogeno"

        elif mod == 'SNaive':
            p = df_test['Prod_t_1'].values
            res_lag, config_str = 1, "Baseline_Naive_Lag1"

        # Proteção Física contra valores negativos de produtividade
        df_res[f'Prev_{mod}'] = np.maximum(p, 0)

        # Avaliação de Métricas (Apenas para anos com dados de validação reais disponíveis)
        rv = df_res['Real'].values
        if not np.isnan(rv).all():
            rmse = np.sqrt(mean_squared_error(rv, p))
            mae = mean_absolute_error(rv, p)
            mape = np.mean(np.abs((rv - p) / np.where(rv == 0, 1, rv))) * 100

            # Tratamento para evitar erro de R2 com poucas amostras locais
            try: r2 = r2_score(rv, p)
            except: r2 = np.nan

            resumo_metricas.append({
                'Ano': ano_alvo, 'Modelo': mod, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2_Agrupado': r2, 'Lag': res_lag
            })

        resumo_hparams.append({'Ano': ano_alvo, 'Modelo': mod, 'Lag_Final': res_lag, 'Config_Detalhada': config_str})

    previsoes_finais.append(df_res)

# ==============================================================================
# SALVAMENTO EM ARQUIVO EXCEL CONSOLIDADO + DOWNLOAD AUTOMÁTICO
# ==============================================================================
nome_arquivo = "RESULTADO_FINAL_ESTATISTICO_sem_chuva.xlsx"

with pd.ExcelWriter(nome_arquivo) as writer:
    if len(previsoes_finais) > 0:
        pd.concat(previsoes_finais).to_excel(writer, sheet_name='Previsoes', index=False)
    if len(resumo_metricas) > 0:
        pd.DataFrame(resumo_metricas).to_excel(writer, sheet_name='Metricas', index=False)
    pd.DataFrame(resumo_hparams).to_excel(writer, sheet_name='Hiperparametros', index=False)

print(f"\n>>> SUCESSO! Dados gravados em '{nome_arquivo}'.")

# Comando exclusivo para o Google Colab baixar o arquivo automaticamente
try:
    from google.colab import files
    print("Iniciando download automático...")
    files.download(nome_arquivo)
except ImportError:
    print(f"Execução local detectada. O arquivo está na pasta: {os.getcwd()}")

--> [Aviso] Ano 2025 estruturado artificialmente para previsão de safra futura.

>>> PROCESSANDO CICLO (JANELA EXPANSIVA): 2023
  .. Executando: LSTM
  .. Executando: LSTM-GRU
  .. Executando: XGBoost
  .. Executando: Random Forest
  .. Executando: Decision Tree
  .. Executando: Prophet


KeyError: "['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm'] not in index"

In [5]:
# ==============================================================================
# BIBLIOTECAS E PACOTES
# ==============================================================================
!pip install -q keras-tuner

import pandas as pd
import numpy as np
import os
import warnings
import re
import shutil
import logging
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# Supressão de logs desnecessários
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

# ==============================================================================
# CONFIGURAÇÕES GLOBAIS
# ==============================================================================
USAR_CHUVA       = False  # Altere para True ou False livremente
MAX_LAG          = 7
CICLOS_VALIDACAO = [2023, 2024, 2025]
GRADE_DEPTH      = [2, 3, 4, 5, 6, 8, 10]

# ==============================================================================
# FUNÇÕES DE CARREGAMENTO E PRÉ-PROCESSAMENTO
# ==============================================================================
def carregar_dados_completo():
    # 1. Carregar Matriz de Produtividade
    df_matriz = pd.read_excel("produtividade_novo_encolding.xlsx")
    col_ano = df_matriz.columns[0]

    col_encoding = [c for c in df_matriz.columns if 'encod' in c.lower() or 'id' in c.lower()]
    col_id_estado = col_encoding[0] if col_encoding else None

    df_longo = df_matriz.melt(id_vars=[col_ano], var_name='estado', value_name='produtividade')
    df_longo.rename(columns={col_ano: 'ano_int'}, inplace=True)

    # Tratar a projeção/estrutura para o ano de 2025
    if 2025 not in df_longo['ano_int'].unique():
        print("--> [Aviso] Ano 2025 estruturado artificialmente para previsão de safra futura.")
        estados = df_longo['estado'].unique()
        df_2025 = pd.DataFrame({'ano_int': [2025]*len(estados), 'estado': estados, 'produtividade': [np.nan]*len(estados)})
        df_base = pd.concat([df_longo, df_2025])
    else:
        print("--> [Sucesso] Dados reais de 2025 detectados na planilha original.")
        df_base = df_longo.copy()

    df_base = df_base.sort_values(['estado', 'ano_int']).reset_index(drop=True)

    # 2. Geração das Variáveis de Lag (Defasagem Temporal)
    for i in range(1, MAX_LAG + 1):
        df_base[f'Prod_t_{i}'] = df_base.groupby('estado')['produtividade'].shift(i)

    # 3. Integração e Tratamento Inteligente dos Dados de Chuva
    if USAR_CHUVA:
        df_c = pd.read_excel("chuva_consolidada_2005_2025_corrigida.xlsx")
        df_c['ano_int'] = df_c['ano'].apply(lambda x: int(re.search(r'(\d{4})', str(x)).group(1)))
        df_c = df_c.groupby(['uf', 'ano_int'])[['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']].mean().reset_index()

        df_base = pd.merge(df_base, df_c.rename(columns={'uf':'estado'}), on=['estado', 'ano_int'], how='left')

        colunas_chuva = ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']
        for col in colunas_chuva:
            df_base[col] = df_base.groupby('estado')[col].transform(lambda x: x.fillna(x.median()))
            df_base[col] = df_base[col].fillna(0)

    # Mapeamento numérico simples para o Estado
    df_base['estado_encoded'] = df_base['estado'].astype('category').cat.codes

    return df_base

# ==============================================================================
# ARQUITETURA DINÂMICA DE REDES NEURAIS (KERAS)
# ==============================================================================
def build_dynamic_model_v42(hp, max_lags, n_clima, hybrid=False):
    n_lags_selected = hp.Int('selected_lags', 2, max_lags)
    inputs = layers.Input(shape=(max_lags + n_clima,))

    def slice_inputs(x):
        lags_part = x[:, :n_lags_selected]
        if n_clima > 0:
            clima_part = x[:, max_lags:]
            return tf.concat([lags_part, clima_part], axis=1)
        return lags_part

    x = layers.Lambda(slice_inputs)(inputs)
    x = layers.Reshape((-1, 1))(x)

    if hybrid:
        x = layers.LSTM(hp.Int('u', 32, 128, step=32), return_sequences=True)(x)
        x = layers.GRU(hp.Int('g', 16, 64, step=16))(x)
    else:
        x = layers.LSTM(hp.Int('u', 32, 128, step=32))(x)

    x = layers.Dropout(hp.Float('drop', 0.0, 0.3, step=0.1))(x)
    outputs = layers.Dense(1)(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('lr', [1e-3, 5e-4])), loss='mse')
    return model

# ==============================================================================
# OTIMIZAÇÃO EM GRADE ANINHADA NO TEMPO (MODELOS TABULARES)
# ==============================================================================
def otimizar_parametros_tabulares_passado(df_historico, ano_alvo, mod, usar_chuva):
    ano_val = ano_alvo - 1
    df_train_val = df_historico[(df_historico['ano_int'] < ano_val) & (df_historico['ano_int'] >= 2013) & (df_historico['produtividade'] > 0)]
    df_test_val  = df_historico[df_historico['ano_int'] == ano_val]

    if len(df_test_val) == 0:
        return 1, 4

    best_l, best_d, best_mae = 1, 4, float('inf')

    for l in range(1, MAX_LAG + 1):
        for d in GRADE_DEPTH:
            f_tab = ['estado_encoded'] + [f'Prod_t_{i}' for i in range(1, l + 1)]
            if usar_chuva:
                f_tab += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

            X_tr_v = df_train_val[f_tab]
            X_ts_v = df_test_val[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=d, random_state=42, n_estimators=100).fit(X_tr_v, df_train_val['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=d, random_state=42, n_jobs=-1, n_estimators=100).fit(X_tr_v, df_train_val['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr_v, df_train_val['produtividade'])

            p_v = m_t.predict(X_ts_v)
            mae_v = mean_absolute_error(df_test_val['produtividade'], p_v)

            if mae_v < best_mae:
                best_mae, best_l, best_d = mae_v, l, d

    return best_l, best_d

# ==============================================================================
# PIPELINE PRINCIPAL DE EXECUÇÃO
# ==============================================================================
df_base = carregar_dados_completo()
n_clima = 3 if USAR_CHUVA else 0
resumo_metricas, resumo_hparams, previsoes_finais = [], [], []
modelos_lista = ['LSTM', 'LSTM-GRU', 'XGBoost', 'Random Forest', 'Decision Tree', 'Prophet', 'SNaive']

for ano_alvo in CICLOS_VALIDACAO:
    print(f"\n>>> PROCESSANDO CICLO (JANELA EXPANSIVA): {ano_alvo}")

    df_train = df_base[(df_base['ano_int'] < ano_alvo) & (df_base['ano_int'] >= 2013) & (df_base['produtividade'] > 0)].copy()
    df_test  = df_base[df_base['ano_int'] == ano_alvo].copy()

    if len(df_test) == 0:
        print(f"  [Aviso] Pulando ciclo {ano_alvo}. Sem dados de validação.")
        continue

    df_res = df_test[['ano_int', 'estado', 'produtividade']].rename(columns={'produtividade': 'Real'})

    feats_full = [f'Prod_t_{i}' for i in range(1, MAX_LAG + 1)]
    if USAR_CHUVA:
        feats_full += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

    sc_x = StandardScaler().fit(df_train[feats_full])
    sc_y = StandardScaler().fit(df_train[['produtividade']])

    X_tr = sc_x.transform(df_train[feats_full])
    X_ts = sc_x.transform(df_test[feats_full])
    y_tr = sc_y.transform(df_train[['produtividade']]).flatten()

    for mod in modelos_lista:
        print(f"  .. Executando: {mod}")

        if mod in ['LSTM', 'LSTM-GRU']:
            dir_tuner = f't_dir_{mod}_{ano_alvo}'
            if os.path.exists(dir_tuner): shutil.rmtree(dir_tuner)

            tuner = kt.RandomSearch(
                lambda hp: build_dynamic_model_v42(hp, MAX_LAG, n_clima, (mod=='LSTM-GRU')),
                objective=kt.Objective('val_loss', direction='min'),
                max_trials=7,
                directory=dir_tuner,
                project_name='tuning'
            )

            callback_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            tuner.search(X_tr, y_tr, epochs=120, validation_split=0.15, callbacks=[callback_early], verbose=0)

            m = tuner.get_best_models(1)[0]
            hp_b = tuner.get_best_hyperparameters(1)[0]
            p = sc_y.inverse_transform(m.predict(X_ts, verbose=0)).flatten()
            res_lag, config_str = hp_b.get('selected_lags'), str(hp_b.values)

            shutil.rmtree(dir_tuner)

        elif mod in ['XGBoost', 'Random Forest', 'Decision Tree']:
            best_l, best_d = otimizar_parametros_tabulares_passado(df_base, ano_alvo, mod, USAR_CHUVA)

            f_tab = ['estado_encoded'] + [f'Prod_t_{i}' for i in range(1, best_l + 1)]
            if USAR_CHUVA:
                f_tab += ['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']

            X_tr_t = df_train[f_tab]
            X_ts_t = df_test[f_tab]

            if mod == 'XGBoost':
                m_t = XGBRegressor(max_depth=best_d, random_state=42, n_estimators=150).fit(X_tr_t, df_train['produtividade'])
            elif mod == 'Random Forest':
                m_t = RandomForestRegressor(max_depth=best_d, random_state=42, n_jobs=-1, n_estimators=150).fit(X_tr_t, df_train['produtividade'])
            else:
                m_t = DecisionTreeRegressor(max_depth=best_d, random_state=42).fit(X_tr_t, df_train['produtividade'])

            p = m_t.predict(X_ts_t)
            res_lag, config_str = best_l, f"Lag_{best_l}_MaxDepth_{best_d}"

        elif mod == 'Prophet':
            dict_p = {}
            # Definindo dinamicamente as colunas do Prophet com base na flag de chuva
            colunas_prophet = ['ano_int', 'produtividade'] + (['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm'] if USAR_CHUVA else [])

            for e in df_test['estado'].unique():
                df_e = df_train[df_train['estado'] == e][colunas_prophet].copy()
                df_e = df_e.rename(columns={'ano_int': 'ds', 'produtividade': 'y'})
                df_e['ds'] = pd.to_datetime(df_e['ds'], format='%Y')

                m_prophet = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
                if USAR_CHUVA:
                    m_prophet.add_regressor('Chuva_Q1_mm')
                    m_prophet.add_regressor('Chuva_Q2_mm')
                    m_prophet.add_regressor('Chuva_Q3_mm')

                m_prophet.fit(df_e)

                df_fut = pd.DataFrame({'ds': [pd.to_datetime(ano_alvo, format='%Y')]})
                if USAR_CHUVA:
                    df_fut_chuva = df_test[df_test['estado'] == e][['Chuva_Q1_mm', 'Chuva_Q2_mm', 'Chuva_Q3_mm']]
                    df_fut = pd.concat([df_fut, df_fut_chuva.reset_index(drop=True)], axis=1)

                pred_p = m_prophet.predict(df_fut)
                dict_p[e] = pred_p['yhat'].values[0]

            p = df_test['estado'].map(dict_p).values
            res_lag, config_str = 0, "Multivariado_Exogeno" if USAR_CHUVA else "Univariado"

        elif mod == 'SNaive':
            p = df_test['Prod_t_1'].values
            res_lag, config_str = 1, "Baseline_Naive_Lag1"

        df_res[f'Prev_{mod}'] = np.maximum(p, 0)

        rv = df_res['Real'].values
        if not np.isnan(rv).all():
            rmse = np.sqrt(mean_squared_error(rv, p))
            mae = mean_absolute_error(rv, p)
            mape = np.mean(np.abs((rv - p) / np.where(rv == 0, 1, rv))) * 100

            try: r2 = r2_score(rv, p)
            except: r2 = np.nan

            resumo_metricas.append({
                'Ano': ano_alvo, 'Modelo': mod, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2_Agrupado': r2, 'Lag': res_lag
            })

        resumo_hparams.append({'Ano': ano_alvo, 'Modelo': mod, 'Lag_Final': res_lag, 'Config_Detalhada': config_str})

    previsoes_finais.append(df_res)

# ==============================================================================
# SALVAMENTO EM ARQUIVO EXCEL CONSOLIDADO + DOWNLOAD AUTOMÁTICO
# ==============================================================================
nome_arquivo = "RESULTADO_FINAL_ESTATISTICO.xlsx"

with pd.ExcelWriter(nome_arquivo) as writer:
    if len(previsoes_finais) > 0:
        pd.concat(previsoes_finais).to_excel(writer, sheet_name='Previsoes', index=False)
    if len(resumo_metricas) > 0:
        pd.DataFrame(resumo_metricas).to_excel(writer, sheet_name='Metricas', index=False)
    pd.DataFrame(resumo_hparams).to_excel(writer, sheet_name='Hiperparametros', index=False)

print(f"\n>>> EXECUÇÃO CONCLUÍDA COM SUCESSO! Dados gravados em '{nome_arquivo}'.")

# Bloco para baixar automaticamente se você estiver usando o Google Colab
try:
    from google.colab import files
    print("Iniciando download automático do Excel...")
    files.download(nome_arquivo)
except ImportError:
    print(f"Execução local detectada. O arquivo está na pasta do script: {os.getcwd()}")

--> [Aviso] Ano 2025 estruturado artificialmente para previsão de safra futura.

>>> PROCESSANDO CICLO (JANELA EXPANSIVA): 2023
  .. Executando: LSTM
  .. Executando: LSTM-GRU
  .. Executando: XGBoost
  .. Executando: Random Forest
  .. Executando: Decision Tree
  .. Executando: Prophet
  .. Executando: SNaive

>>> PROCESSANDO CICLO (JANELA EXPANSIVA): 2024
  .. Executando: LSTM
  .. Executando: LSTM-GRU
  .. Executando: XGBoost
  .. Executando: Random Forest
  .. Executando: Decision Tree
  .. Executando: Prophet
  .. Executando: SNaive

>>> PROCESSANDO CICLO (JANELA EXPANSIVA): 2025
  .. Executando: LSTM
  .. Executando: LSTM-GRU
  .. Executando: XGBoost
  .. Executando: Random Forest
  .. Executando: Decision Tree
  .. Executando: Prophet
  .. Executando: SNaive

>>> EXECUÇÃO CONCLUÍDA COM SUCESSO! Dados gravados em 'RESULTADO_FINAL_ESTATISTICO.xlsx'.
Iniciando download automático do Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>